# Managed MCP: BigQuery Analyst Demo (February 2026 Preview)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/maruti123/partner-demos/blob/main/mcp_bigquery_demo.ipynb)

This notebook demonstrates the **Managed MCP (Model Context Protocol)** feature in Google Cloud. It connects an ADK Agent to a Managed BigQuery MCP server to perform natural language data analysis.

## Use Case
A data analyst needs to query BigQuery tables without writing SQL. The agent uses the BigQuery MCP tool to discover schemas and execute queries on behalf of the user.

### Requirements
- `google-adk >= 1.26.0` installed.
- BigQuery API and Managed MCP service enabled.

In [ ]:
# 1. Setup and Environment
# !pip install google-adk google-genai nest-asyncio
# from google.colab import auth
# auth.authenticate_user()

import os
import nest_asyncio
nest_asyncio.apply()

project_id = 'YOUR_PROJECT_ID'  # @param {type:"string"}
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "TRUE"
os.environ["GOOGLE_CLOUD_PROJECT"] = project_id
os.environ["GOOGLE_CLOUD_LOCATION"] = "us-central1"

### 2. [MANDATORY] Enable the Managed BigQuery MCP Service

Managed MCP servers must be explicitly enabled for your project.

In [ ]:
# Enable the BigQuery MCP service
!gcloud config set project {project_id}
!gcloud beta services mcp enable bigquery.googleapis.com --project={project_id}
print("Success: Managed BigQuery MCP service enabled.")

### 3. [PREREQUISITES] Create Dummy Dataset and Table

Ensure the agent has data to analyze.

In [ ]:
# 3. Data Setup
from google.cloud import bigquery

def setup_demo_data():
    client = bigquery.Client(project=project_id)
    dataset_id = f"{project_id}.us_sales"
    
    # Create Dataset
    dataset = bigquery.Dataset(dataset_id)
    dataset.location = "us-central1"
    client.create_dataset(dataset, exists_ok=True)

    # Create Table
    table_id = f"{dataset_id}.sales_data"
    schema = [
        bigquery.SchemaField("product_id", "STRING"),
        bigquery.SchemaField("units_sold", "INTEGER"),
        bigquery.SchemaField("sale_date", "DATE"),
    ]
    table = bigquery.Table(table_id, schema=schema)
    client.create_table(table, exists_ok=True)
    
    # Insert Dummy Data
    client.insert_rows_json(table_id, [
        {"product_id": "PROD_001", "units_sold": 100, "sale_date": "2026-03-16"}
    ])
    print(f"Demo data prepared at {table_id}")

setup_demo_data()

### 4. Initialize the BigQuery MCP Tool and Agent

We use **Gemini 3.1 Pro** and the **ADK McpToolset** with a `DummyStream` to handle Colab's I/O limitations.

In [ ]:
# 4. Initialize Toolset and Agent
import sys
from google.adk import Agent, Runner
from google.adk.tools.mcp_tool.mcp_toolset import McpToolset, StdioConnectionParams
from google.adk.tools.mcp_tool.mcp_session_manager import StdioServerParameters
from google.adk.sessions.in_memory_session_service import InMemorySessionService
import os
import asyncio
import google.auth
from google.auth.transport.requests import Request
from google.adk.agents import LlmAgent
from google.adk.runners import InMemoryRunner
# Notice the updated spelling: McpToolset
from google.adk.tools.mcp_tool import McpToolset, StreamableHTTPConnectionParams



# Initialize the BigQuery MCP toolset using recommended ConnectionParams
scopes = ["https://www.googleapis.com/auth/cloud-platform"]
creds, _ = google.auth.default(scopes=scopes)
creds.refresh(Request())

# 3. Configure the ADK MCP Toolset (Using the new class name)
bq_mcp_toolset = McpToolset(
    connection_params=StreamableHTTPConnectionParams(
        url="https://bigquery.googleapis.com/mcp",
        headers={
            "Authorization": f"Bearer {creds.token}",
            "Content-Type": "application/json",
            "x-goog-user-project": os.environ["GOOGLE_CLOUD_PROJECT"]
        },
        timeout=30.0 
    )
)

agent = Agent(
    model="gemini-3.1-pro",
    name="BigQueryAssistant",
    instruction="You are a helpful data analyst. Use your BigQuery tools to answer user questions.",
    tools=[bq_mcp_toolset]
)

runner = Runner(
    agent=agent, 
    session_service=InMemorySessionService(),
    app_name="MCPDemoApp",
    auto_create_session=True
)
print("Agent and BigQuery MCP Toolset ready.")

### 5. Execute a Natural Language Query

The agent will automatically use the MCP tool to discover schemas and run the SQL.

In [ ]:
print("Agent initialized via Vertex AI. Processing request...\n")
user_prompt = "Can you list the datasets available in my BigQuery project? YOUR_PROJECT_ID" 

print("--- Agent Response ---")
await runner.run_debug(user_prompt, verbose=True)

### 6. Things to remember or know
- **Zero Boilerplate**: MCP removes the need to write custom SQL functions for the agent.
- **Managed Security**: The BigQuery MCP server respects your IAM permissions natively.
- **Gemini 3.1 Pro**: Optimized for complex data reasoning and multi-step tool use via MCP.